# Card effect modeling corpus audit

Recomputes the key coverage and semantic-inventory statistics used by `docs/CARD-EFFECT-MODELING.md`. The notebook uses only the Python standard library and treats free-text classifications as triage metadata, never executable game rules.

In [ ]:
import json
from collections import Counter
from pathlib import Path

repo = Path.cwd()
if not (repo / 'artifacts' / 'card-modeling' / 'current').is_dir():
    repo = repo.parent
root = repo / 'artifacts' / 'card-modeling' / 'current'
manifest = json.loads((root / 'corpus-manifest.json').read_text(encoding='utf-8-sig'))
cards = json.loads((root / 'mainstream-cards.json').read_text(encoding='utf-8-sig'))
semantics_root = json.loads((root / 'mainstream-card-semantics.json').read_text(encoding='utf-8-sig'))
semantics = semantics_root['cards']
summary = json.loads((root / 'semantic-summary.json').read_text(encoding='utf-8-sig'))
len(cards), len(semantics)

In [ ]:
assert manifest['counts']['unique_cards'] == 295
assert len(cards) == len(semantics) == 295
assert len({card['card_id'] for card in cards}) == 295
assert {card['card_id'] for card in cards} == {card['card_id'] for card in semantics}
assert all(isinstance(card['mechanics'], list) for card in cards)
assert all(card['semantic_inventory']['operations'] for card in semantics)
assert summary['scope']['meta_share_pct'] == 88.45
'integrity checks passed'

In [ ]:
readiness = Counter(card['modeling']['execution_readiness'] for card in semantics)
coverage = {
    'selected_decks': manifest['counts']['selected_decks'],
    'core_evidence_decks': manifest['counts']['core_evidence_decks'],
    'deck_card_rows': manifest['counts']['deck_card_rows'],
    'unique_cards': len(semantics),
    'existing_verified_rules': readiness['existing_verified_rule'],
    'template_candidates': readiness['template_candidate_requires_review'],
    'manual_ir_required': readiness['manual_ir_rule_required'],
}
coverage

In [ ]:
def axis_counts(path):
    values = Counter()
    for card in semantics:
        current = card
        for key in path:
            current = current[key]
        values.update(current)
    return values

top_operations = axis_counts(('semantic_inventory', 'operations')).most_common(10)
modeling_families = axis_counts(('modeling', 'families')).most_common()
top_operations, modeling_families

In [ ]:
risk_counts = {
    'stochastic_or_choice': sum(bool(card['semantic_inventory']['stochasticity']) for card in semantics),
    'hidden_information': sum(bool(card['semantic_inventory']['hidden_information']) for card in semantics),
    'history_dependency': sum(bool(card['semantic_inventory']['history_dependencies']) for card in semantics),
    'dynamic_or_variant_text': sum(bool(card['modeling']['text_quality_flags']) for card in semantics),
    'referenced_game_tags': sum(bool(card['referenced_game_tags']) for card in semantics),
    'entourage_card_ids': sum(bool(card['semantic_inventory']['entourage_card_ids']) for card in semantics),
}
assert risk_counts == {
    'stochastic_or_choice': 91,
    'hidden_information': 59,
    'history_dependency': 68,
    'dynamic_or_variant_text': 25,
    'referenced_game_tags': 72,
    'entourage_card_ids': 0,
}
risk_counts

## Interpretation

The 295-card mainstream slice is a prioritization corpus, not a complete ruleset. A complete Standard registry starts from all 1,152 cards and then closes over tokens, enchantments, hero powers, forms, rewards, and generated pools. The empty Entourage result demonstrates that CardDefs does not provide that dependency graph by itself.